# メッシュ農業気象データ・ダウンロード
- インポートのセルと、前準備のセルは必ず実行する
- それ以外のデータダウンロード処理・データ結合処理は、各章ごとに個別に実行可能
- データ結合を実行する場合は、カーネルを再起動して、インポートのセルと前準備のセルを再度実行することが望ましい

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from lib import AMD_Tools4 as amd

## 必要なデータダウンロード期間

### トップリバー

In [ ]:
#--データ読み込み
farm_dir = 'farm'
farm_excel = '【Confidential】サンプルデータ（トップリバー様） (2).xlsx'
excel_sheet = 'トップリバー様（NDVI）'

data = pd.read_excel(f'{farm_dir}/{farm_excel}', sheet_name=excel_sheet)
data.dropna(inplace=True)
data['定植日'] = pd.to_datetime(data['定植日'])
data['収穫日'] = pd.to_datetime(data['収穫日'])

In [ ]:
#--各年の最初の定植日と最後の収穫日
for y, df in data.groupby(data['定植日'].dt.year):
    print(y, df['定植日'].sort_values().values[0], df['収穫日'].sort_values(ascending=False).values[0])

### カヤノ農産

In [ ]:
#--データ読み込み
farm_dir = 'farm'
farm_excel = '【confidential】サンプルデータ　カヤノ農産様.xlsx'
excel_sheet = 'カヤノ農産様　データ'

data = pd.read_excel(f'{farm_dir}/{farm_excel}', sheet_name=excel_sheet)
data.dropna(inplace=True)
data['定植日'] = pd.to_datetime(data['定植日'])
data['収穫日'] = pd.to_datetime(data['収穫日'])

In [ ]:
#--最初と最後の年を確認
print(data['定植日'].sort_values().values[0], data['収穫日'].sort_values(ascending=False).values[0])

### ジャパンアグリ

In [ ]:
#--データ読み込み
farm_dir = 'farm'
farm_excel = '【confidential】サンプルデータ　ジャパンアグリ様②.xlsx'
excel_sheet = 'Sheet1'

data = pd.read_excel(f'{farm_dir}/{farm_excel}', sheet_name=excel_sheet)
data.dropna(inplace=True)
data['定植日'] = pd.to_datetime(data['定植日'])
data['収穫日'] = pd.to_datetime(data['収穫日'])

In [ ]:
#--各年の最初の定植日と最後の収穫日
for y, df in data.groupby(data['定植日'].dt.year):
    print(y, df['定植日'].sort_values().values[0], df['収穫日'].sort_values(ascending=False).values[0])

- トップリバー
    - 2018年〜2025年
    - 概ね毎年、2月10日〜11月10日の間
- カヤノ農産
    - 2021年〜2025年
    - 基本的に通年
- ジャパンアグリ
    - 2023年〜2025年
    - 概ね毎年、1月〜11月

→ であれば、1年通してデータを取得しておく

## データダウンロード実行

### 前準備
- グローバル変数の定義

In [2]:
#--グローバル変数設定
class Global_Parameters:
    #--圃場リスト
    farm_topriver = "farm/topriver_farm.csv"
    farm_kayano = "farm/kayano_farm.csv"
    farm_jpagri = "farm/jpagri_farm.csv"
    #--圃場リスト（気象データ取得用）
    farm_amd_topriver = "farm/topriver_farm_amd.csv"
    farm_amd_kayano = "farm/kayano_farm_amd.csv"
    farm_amd_jpagri = "farm/jpagri_farm_amd.csv"
    #--出力先ディレクトリ
    output_dir = "weather"
    #--気象要素リスト
    AMD_elements = [
        'TMP_mea','TMP_max','TMP_min',
        'APCP','APCPRA',
        'SSD','GSR','DLR',
        'RH','WIND',
        'SD','SWE','SFW'
    ]
    #--気象データ取得期間
    # range_topriver = pd.date_range(start="2018-01-01", end="2025-11-01", freq="ME")
    # range_kayano = pd.date_range(start="2021-01-01", end="2025-11-01", freq="ME")
    # range_jpagri = pd.date_range(start="2022-01-01", end="2025-11-01", freq="ME")
    range_topriver = pd.date_range(start="2025-11-01", end="2025-12-01", freq="ME")
    range_kayano = pd.date_range(start="2025-11-01", end="2025-12-01", freq="ME")
    range_jpagri = pd.date_range(start="2025-11-01", end="2025-12-01", freq="ME")

gp = Global_Parameters()

### AMDの座標情報を取る
- 1km以内で近接した圃場も多いため、どの緯度経度の格子点値が取得されるか、あらかじめテストして情報を取っておく
- 一度実行してCSV出力しておけば、次回以降は気象データダウンロードに進んでよい

In [ ]:
#--圃場ごとのAMD座標情報を取得
def Check_AMD_LatLon(df_field):
    time_domain = ['2025-01-01', '2025-01-02']
    element = "TMP_mea"

    list_lat = []
    list_lon = []
    for fid in df_field.index:
        lat = df_field.loc[fid,'latitude']
        lon = df_field.loc[fid,'longitude']
        latlon_domain = [lat, lat, lon, lon]
        data, time_array, lat_array, lon_array = amd.GetMetData(element, time_domain, latlon_domain)
        list_lat.append(lat_array[0])
        list_lon.append(lon_array[0])

    df_amdinfo = pd.DataFrame({'amd_lat':list_lat, 'amd_lon':list_lon}, index=df_field.index)
    return df_amdinfo

In [ ]:
#--地点リスト読み込み
topriver = pd.read_csv(gp.farm_topriver).set_index('field_id')
kayano = pd.read_csv(gp.farm_kayano).set_index('field_id')
jpagri = pd.read_csv(gp.farm_jpagri).set_index('field_id')

In [ ]:
#--圃場ごとのAMD座標情報を取得し、CSV出力
topriver = pd.concat([topriver, Check_AMD_LatLon(topriver)], axis=1)
topriver.to_csv(gp.farm_amd_topriver)

kayano = pd.concat([kayano, Check_AMD_LatLon(kayano)], axis=1)
kayano.to_csv(gp.farm_amd_kayano)

jpagri = pd.concat([jpagri, Check_AMD_LatLon(jpagri)], axis=1)
jpagri.to_csv(gp.farm_amd_jpagri)

### 気象データ（過去の解析データ）

In [3]:
#--メッシュ農業気象データの取得を実行する関数
def AMD_Data(farm_name, list_date, df_field, cli=False):
    df_field = df_field.reset_index()
    df_unique = df_field.drop_duplicates(subset=['amd_lat','amd_lon']).reset_index()
    df_unique['amd_id'] = ["amd" + str(i).zfill(3) for i in range(len(df_unique))]
    df_field = pd.merge(df_field, df_unique[['amd_lat','amd_lon','amd_id']], on=['amd_lat','amd_lon'], how="left")
    df_unique = df_unique.set_index('amd_id')

    #--年月のループ
    for date in list_date:
        time_domain = [f'{date.year}-{date.month:02d}-01', f'{date.year}-{date.month:02d}-{date.day:02d}']
        time_range = pd.date_range(*time_domain, freq="D").date

        #--気象要素のループ
        for element in gp.AMD_elements:
            output_path = f"{gp.output_dir}/{farm_name}/{element}"
            os.makedirs(output_path, exist_ok=True)

            #--圃場のループ
            values = None
            for fid in df_unique.index:
                lat = df_unique.loc[fid,'amd_lat']
                lon = df_unique.loc[fid,'amd_lon']

                #--メッシュ農業気象データ取得
                latlon_domain = [lat, lat, lon, lon]
                data, time_array, lat_array, lon_array = amd.GetMetData(element, time_domain, latlon_domain, cli=cli)
                if values is None:
                    values = data[:,0,0].reshape(-1,1)
                else:
                    values = np.concatenate([values, data[:,0,0].reshape(-1,1)], axis=1)

            #--データフレームとして整理してCSV出力
            df_wxdata = pd.DataFrame(values.astype(np.float32), index=time_range, columns=df_unique.index)
            df_output = df_wxdata[ df_field['amd_id'].to_list() ]
            df_output.columns = df_field['field_id'].to_list()

            output_file = f"{farm_name}_{element}_{date.year}{date.month:02d}.csv"
            df_output.reset_index(names='date').to_csv(f"{output_path}/{output_file}", index=False)

In [4]:
#--AMD座標情報を持つ地点リスト読み込み
topriver = pd.read_csv(gp.farm_amd_topriver, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')
kayano = pd.read_csv(gp.farm_amd_kayano, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')
jpagri = pd.read_csv(gp.farm_amd_jpagri, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')

In [5]:
#--メッシュ農業気象データ取得実行
AMD_Data("topriver", gp.range_topriver, topriver)
AMD_Data("kayano", gp.range_kayano, kayano)
AMD_Data("jpagri", gp.range_jpagri, jpagri)

TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_mea (30, 1, 1) Tile
TMP_max (30, 1, 1) Tile
TMP_max (30, 1, 1) Tile
TMP_max (30, 1, 1) Tile
TMP_max (30, 1, 

- データ取得には時間がかかる（約半日）
- ネットワーク接続が切れるなどの理由でデータ取得に失敗する場合がある（再トライする）

### 平年値
- 通常年かうるう年かに関わらず、同じ日付であれば同じ値になっている
- よって2024年の1年間でデータを取得する

In [ ]:
#--データ取得関数は、上記気象データ取得と同じものを使用する

In [ ]:
#--グローバル変数の一部修正
gp.output_dir = "climate"
gp.AMD_elements = ['TMP_mea','TMP_max','TMP_min','APCP','APCPRA','SSD','GSR']
gp.range_topriver = pd.date_range(start="2024-01-01", end="2025-01-01", freq="ME")
gp.range_kayano = pd.date_range(start="2024-01-01", end="2025-01-01", freq="ME")
gp.range_jpagri = pd.date_range(start="2024-01-01", end="2025-01-01", freq="ME")

In [ ]:
#--AMD座標情報を持つ地点リスト読み込み
topriver = pd.read_csv(gp.farm_amd_topriver, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')
kayano = pd.read_csv(gp.farm_amd_kayano, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')
jpagri = pd.read_csv(gp.farm_amd_jpagri, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')

In [ ]:
#--平年値取得実行
AMD_Data("topriver", gp.range_topriver, topriver, cli=True)
AMD_Data("kayano", gp.range_kayano, kayano, cli=True)
AMD_Data("jpagri", gp.range_jpagri, jpagri, cli=True)

### 土地利用情報
- 標高、および土地利用率（田畑、森林、荒地など）を取得

In [ ]:
#--土地利用情報を取得する関数
def Geo_Data(farm_name, df_field):
    df_field = df_field.reset_index()
    df_unique = df_field.drop_duplicates(subset=['amd_lat','amd_lon']).reset_index()
    df_unique['amd_id'] = ["amd" + str(i).zfill(3) for i in range(len(df_unique))]
    df_field = pd.merge(df_field, df_unique[['amd_lat','amd_lon','amd_id']], on=['amd_lat','amd_lon'], how="left")
    df_unique = df_unique.set_index('amd_id')

    #--気象要素のループ
    for element in gp.AMD_elements:
        output_path = f"{gp.output_dir}/{farm_name}/{element}"
        os.makedirs(output_path, exist_ok=True)

        #--圃場のループ
        values = None
        for fid in df_unique.index:
            lat = df_unique.loc[fid,'amd_lat']
            lon = df_unique.loc[fid,'amd_lon']

            #--メッシュ農業気象データ取得
            latlon_domain = [lat, lat, lon, lon]
            data, lat_array, lon_array = amd.GetGeoData(element, latlon_domain)
            if values is None:
                values = data[:,0].reshape(-1,1)
            else:
                values = np.concatenate([values, data[:,0].reshape(-1,1)], axis=1)

        #--データフレームとして整理してCSV出力
        df_wxdata = pd.DataFrame(values.astype(np.float32), columns=df_unique.index)
        df_output = df_wxdata[ df_field['amd_id'].to_list() ]
        df_output.columns = df_field['field_id'].to_list()

        output_file = f"{farm_name}_{element}.csv"
        df_output.to_csv(f"{output_path}/{output_file}", index=False)

In [ ]:
#--グローバル変数の一部修正
gp.output_dir = "geo"
gp.AMD_elements = [
    'altitude',
    'landuse_H210100','landuse_H210200','landuse_H210500','landuse_H210600',
    'landuse_H210700','landuse_H210901','landuse_H210902','landuse_H211000',
    'landuse_H211100','landuse_H211600'
]

In [ ]:
#--AMD座標情報を持つ地点リスト読み込み
topriver = pd.read_csv(gp.farm_amd_topriver, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')
kayano = pd.read_csv(gp.farm_amd_kayano, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')
jpagri = pd.read_csv(gp.farm_amd_jpagri, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')

In [ ]:
#--地理情報取得実行
Geo_Data("topriver", topriver)
Geo_Data("kayano", kayano)
Geo_Data("jpagri", jpagri)

- カヤノ農産の圃場の土地利用情報は、その他農用地ではなく、半分以上が森林で残りも荒地になっている
- トップリバーの圃場も、必ずしもその他農用地が占める割合が大きくない

### データ結合
- 気象要素ごと、1ヶ月ごとにダウンロードしたデータを、圃場ごとに整理してCSV出力（気象値・平年値）
- 地理情報はまとめるほどのファイル数ではないため、そのまま使用することにする

In [6]:
#--AMD座標情報を持つ地点リスト読み込み
topriver = pd.read_csv(gp.farm_amd_topriver, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')
kayano = pd.read_csv(gp.farm_amd_kayano, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')
jpagri = pd.read_csv(gp.farm_amd_jpagri, dtype={'amd_lat':np.float32, 'amd_lon':np.float32}).set_index('field_id')

In [7]:
#--圃場ごとにデータを収集・整理してCSV保存する関数
def Make_Field_Weather(farm_name, df_field):
    #--気象要素ごとに全データを縦結合
    df_dict = {}
    for element in gp.AMD_elements:
        files = glob.glob(f"{gp.output_dir}/{farm_name}/{element}/*.csv")

        all_df = []
        for file in files:
            df = pd.read_csv(file)
            all_df.append(df)
        df_dict[element] = pd.concat(all_df, axis=0).sort_values('date').set_index('date')

    #--圃場ごとに整理してCSV出力
    for fid in df_field.index:
        my_df = pd.DataFrame()
        for element in gp.AMD_elements:
            my_df = pd.concat([my_df, df_dict[element][[fid]].rename(columns={fid:element})], axis=1)

        output_file = f"{gp.output_dir}/{farm_name}/{gp.output_dir}_{fid}.csv"
        my_df.to_csv(output_file)

In [8]:
### 気象データ ###
#--データ整理実行
Make_Field_Weather("topriver", topriver)
Make_Field_Weather("kayano", kayano)
Make_Field_Weather("jpagri", jpagri)

In [ ]:
### 平年値データ ###
#--データ整理実行
gp.output_dir = "climate"
gp.AMD_elements = ['TMP_mea','TMP_max','TMP_min','APCP','APCPRA','SSD','GSR']
Make_Field_Weather("topriver", topriver)
Make_Field_Weather("kayano", kayano)
Make_Field_Weather("jpagri", jpagri)

In [ ]:
#--圃場ごとにデータを収集・整理してCSV保存する関数
def Make_Field_Geo(farm_name, df_field):
    #--要素ごとに全データ読み込み
    df_dict = {}
    for element in gp.AMD_elements:
        file = f"{gp.output_dir}/{farm_name}/{element}/{farm_name}_{element}.csv"
        df = pd.read_csv(file)
        df_dict[element] = df

    #--圃場ごとに整理してCSV出力
    all_df = pd.DataFrame()
    for fid in df_field.index:
        my_df = pd.DataFrame({'field_id':[fid]})
        for element in gp.AMD_elements:
            my_df = pd.concat([my_df, df_dict[element][[fid]].rename(columns={fid:element})], axis=1)
        all_df = pd.concat([all_df, my_df], axis=0)

    output_file = f"{gp.output_dir}/{farm_name}/{gp.output_dir}_{farm_name}.csv"
    all_df.to_csv(output_file, index=False)

In [ ]:
### 地理情報データ ###
#--データ整理実行
gp.output_dir = "geo"
gp.AMD_elements = [
    'altitude',
    'landuse_H210100','landuse_H210200','landuse_H210500','landuse_H210600',
    'landuse_H210700','landuse_H210901','landuse_H210902','landuse_H211000',
    'landuse_H211100','landuse_H211600'
]
Make_Field_Geo("topriver", topriver)
Make_Field_Geo("kayano", kayano)
Make_Field_Geo("jpagri", jpagri)

## コード一時保存

In [ ]:
### このやり方は時間がかかるため、上記のやり方に変更した ###
# #--メッシュ農業気象データの取得を実行する関数（1地点ごと）
# def AMD_Data(farm_name, list_date, df_field):
#     #--年月のループ
#     for date in list_date:
#         time_domain = [f'{date.year}-{date.month:02d}-01', f'{date.year}-{date.month:02d}-{date.day:02d}']
#         time_range = pd.date_range(*time_domain, freq="D").date

#         #--気象要素のループ
#         for element in gp.AMD_elements:
#             output_path = f"{gp.output_dir}/{farm_name}/{element}"
#             os.makedirs(output_path, exist_ok=True)

#             #--圃場のループ
#             values = None
#             for fid in df_field.index:
#                 lat = df_field.loc[fid,'latitude']
#                 lon = df_field.loc[fid,'longitude']

#                 #--メッシュ農業気象データ取得
#                 latlon_domain = [lat, lat, lon, lon]
#                 data, time_array, lat_array, lon_array = amd.GetMetData(element, time_domain, latlon_domain)
#                 if values is None:
#                     values = data[:,0,0].reshape(-1,1)
#                 else:
#                     values = np.concatenate([values, data[:,0,0].reshape(-1,1)], axis=1)

#             #--データフレームとして整理してCSV出力
#             df_output = pd.DataFrame(values.astype(np.float32), index=time_range, columns=df_field.index)
#             output_file = f"{farm_name}_{element}_{date.year}{date.month:02d}.csv"
#             df_output.reset_index(names='date').to_csv(f"{output_path}/{output_file}", index=False)